# Multiple testing and cost stress: deterministic example

Use three fixed p-values to see Holm adjustment. Then check the effect of two cost schedules on a single synthetic gross cashflow held fixed.

In [1]:
from math import isclose

raw_p = {"primary_example": 0.02, "secondary_a": 0.03, "secondary_b": 0.20}
ordered = sorted(raw_p, key=raw_p.get)
adjusted = {}
running_maximum = 0.0
for rank, name in enumerate(ordered):
    weighted = (len(ordered) - rank) * raw_p[name]
    running_maximum = max(running_maximum, min(1.0, weighted))
    adjusted[name] = running_maximum

assert all(
    isclose(adjusted[name], expected)
    for name, expected in zip(ordered, (0.06, 0.06, 0.20), strict=True)
)

midpoint_gross = 18
costs = {
    "reference": {"spread": 3, "slippage": 1, "commission": 2},
    "stressed": {"spread": 5, "slippage": 4, "commission": 2},
}
recorded_net = {"reference": 12, "stressed": 7}
residuals = {
    name: midpoint_gross - sum(charges.values()) - recorded_net[name]
    for name, charges in costs.items()
}
assert all(residual == 0 for residual in residuals.values())
print("Synthetic adjusted p-values:", adjusted)
print("Synthetic cost reconciliation residuals:", residuals)

Synthetic adjusted p-values: {'primary_example': 0.06, 'secondary_a': 0.06, 'secondary_b': 0.2}
Synthetic cost reconciliation residuals: {'reference': 0, 'stressed': 0}


Fixed-fill arithmetic explains the cashflow identity. In a research stress replay, quote and cost changes can affect which orders fill and whether the account completes its path.